# LLM repair benchmark (Claude vs. Horizon)

Repairs every injected table in `datasets/hospital_2k/` and
`datasets/insurance_claims_2k/` with **Claude** (via the Claude Code CLI in
headless mode, using your Pro/Max **subscription** &mdash; no API key), then
scores each repair against `clean.csv` with the same &sect;6.1 metrics as
`repair_eval.ipynb`, so LLM quality is directly comparable to Horizon's.

**Per (table, run):** the model reads the dirty table + FDs and returns a JSON
**edit list** (only the cells it changes); the notebook applies those edits into
a copy of the dirty table to build the `cleaned` table, then scores it. Each
table is repaired `N_RUNS` times (fresh context each time) so F1 can be averaged
across runs. All the model logic lives in `benchmarks/llm_repair.py`.

**Prereqs:** the `claude` CLI installed and logged in to your subscription.
Sanity-check the mechanism first:

```
echo "reply with the word ok" | claude -p --model sonnet --allowedTools ""
```

**Cost note:** this is ~32 tables &times; `N_RUNS` calls, each with a large
(~120K-token) prompt &mdash; it uses real subscription quota and takes a while.
The run cell is **resumable** (skips `(dataset, table, run)` already in
`results.csv`), so start small (one dataset / `N_RUNS = 1`) to validate, then
scale up.

**Inputs → Outputs:** `datasets/{hospital_2k, insurance_claims_2k}/…` →
`experiments/llm_repair/results.csv` (one row per (dataset, table, run)) and
`per_table_f1.csv`; cleaned tables land under `experiments/llm_repair/<dataset>/`.

**Config knobs:**
- `DATABASES` — dataset folders to benchmark.
- `MODEL` — Claude alias passed to `claude --model` (sonnet / opus / haiku).
- `N_RUNS` — repairs per table (raise to average F1 / see variance).
- `TIMEOUT`, `SLEEP` — per-call CLI timeout and inter-call pause.

## Setup — imports & sys.path

In [ ]:
import logging
import sys
import time
from pathlib import Path

import polars as pl

# Put the repo root on sys.path so `horizon`, `eval`, and `benchmarks` import as
# packages (mirrors notebooks/repair_eval.ipynb). horizon/ is also added for the
# pipeline's flat imports; harmless here.
REPO = Path("..").resolve()
HZN = REPO / "horizon"
for p in (str(HZN), str(REPO)):
    if p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(HZN))
sys.path.insert(0, str(REPO))

from horizon.utils.loaders import load_fds, load_table
from eval.effectiveness_eval import evaluate_repair
from benchmarks.llm_repair import repair_one

DATASETS = REPO / "datasets"
OUTPUT = REPO / "experiments" / "llm_repair"
OUTPUT.mkdir(parents=True, exist_ok=True)

logging.getLogger("horizon").setLevel(logging.WARNING)

## Configuration

In [ ]:
# Which datasets to benchmark (folders under datasets/).
DATABASES = ["hospital_2k", "insurance_claims_2k"]

# Model alias passed to `claude --model` (subscription auth). "sonnet" -> Sonnet 5;
# switch to "opus" to rerun on Opus 4.8, or "haiku" for a low-end baseline.
MODEL = "sonnet"

N_RUNS = 1     # repairs per table (1 = one pass; raise to average F1 / see variance)
TIMEOUT = 1200  # per-call CLI timeout (seconds); large tables can take minutes
SLEEP = 2.0    # pause between calls, to be gentle on rate limits

# One result row per (dataset, table, run); fixed schema so resume/append is clean.
COLUMNS = [
    "dataset", "table", "etype", "rate", "level", "run", "model", "status",
    "n_dirty", "n_repaired", "n_correct", "precision", "recall", "f1",
    "n_edits", "n_invalid", "secs", "cost", "error",
]
RESULTS = OUTPUT / "results.csv"

## Run the benchmark (resumable)

Repairs each `(dataset, table, run)` not already in `results.csv` and appends its
score; the whole file is rewritten after every call so an interrupted run resumes.

In [ ]:
def parse_stem(stem):
    """injected stem -> (etype, rate, level): 'e1_r05_high' -> ('e1', 5, 'high')."""
    etype, _, tail = stem.partition("_r")
    rate_str, _, level = tail.partition("_")
    return etype, int(rate_str), (level or "")


def persist(rows):
    # write every field as a string so resumed (all-Utf8) and new rows never clash
    pl.DataFrame(
        [{k: ("" if r.get(k) is None else str(r.get(k))) for k in COLUMNS} for r in rows]
    ).write_csv(RESULTS)


# resume: keep prior rows, skip (dataset, table, run) already recorded
rows, done = [], set()
if RESULTS.exists():
    for r in pl.read_csv(RESULTS, infer_schema_length=0).to_dicts():
        rows.append(r)
        done.add((r["dataset"], r["table"], int(r["run"])))

for database in DATABASES:
    ds_dir = DATASETS / database
    clean_path = ds_dir / "clean.csv"
    set_of_fds = load_fds(ds_dir, clean_path)
    clean = load_table(clean_path)
    out_dir = OUTPUT / database
    injected = sorted((ds_dir / "injected").glob("*_r*.csv"))
    assert injected, f"no injected/*_r*.csv under {ds_dir}"

    for path in injected:
        etype, rate, level = parse_stem(path.stem)
        dirty = load_table(path)
        for run in range(N_RUNS):
            if (database, path.stem, run) in done:
                print(f"skip {database}/{path.stem} run{run} (done)")
                continue

            meta = repair_one(path, set_of_fds, MODEL, out_dir, run, timeout=TIMEOUT)
            row = {c: None for c in COLUMNS}
            row.update(
                dataset=database, table=path.stem, etype=etype, rate=rate,
                level=level, run=run, model=MODEL, status=meta["status"],
                secs=round(meta.get("secs", 0.0), 2), cost=meta.get("cost"),
                error=meta.get("error", ""),
            )
            if meta["status"] == "ok":
                m = evaluate_repair(clean, dirty, load_table(Path(meta["cleaned_path"])))
                row.update(
                    n_dirty=m["n_dirty"], n_repaired=m["n_repaired"],
                    n_correct=m["n_correct"], precision=round(m["precision"], 4),
                    recall=round(m["recall"], 4), f1=round(m["f1"], 4),
                    n_edits=meta["n_edits"], n_invalid=meta["n_invalid"],
                )
                print(f"{database}/{path.stem:24s} run{run}  F1={row['f1']:.3f}")
            else:
                print(f"{database}/{path.stem:24s} run{run}  {meta['status']}: {str(meta.get('error',''))[:80]}")

            rows.append(row)
            done.add((database, path.stem, run))
            persist(rows)   # crash-safe: whole file rewritten after each call
            time.sleep(SLEEP)

print(f"\ndone; {len(rows)} rows -> {RESULTS}")

## Aggregate results — mean F1 per dataset and per table

In [ ]:
res = pl.read_csv(RESULTS, infer_schema_length=0)

print("runs by status:")
print(res.group_by("status").len().sort("status"))

ok = res.filter(pl.col("status") == "ok").with_columns(
    pl.col("rate").cast(pl.Int64),
    pl.col("f1").cast(pl.Float64),
    pl.col("precision").cast(pl.Float64),
    pl.col("recall").cast(pl.Float64),
)

per_table = (
    ok.group_by(["dataset", "table", "etype", "rate", "level"])
    .agg(
        pl.col("f1").mean().round(4).alias("f1_mean"),
        pl.col("f1").std().round(4).alias("f1_std"),
        pl.col("precision").mean().round(4).alias("precision_mean"),
        pl.col("recall").mean().round(4).alias("recall_mean"),
        pl.len().alias("n_runs"),
    )
    .sort(["dataset", "etype", "rate", "level"])
)

per_dataset = (
    ok.group_by("dataset")
    .agg(
        pl.col("f1").mean().round(4).alias("f1_mean"),
        pl.col("precision").mean().round(4).alias("precision_mean"),
        pl.col("recall").mean().round(4).alias("recall_mean"),
        pl.len().alias("n_runs"),
    )
    .sort("dataset")
)

per_table.write_csv(OUTPUT / "per_table_f1.csv")
print("\nper-dataset mean F1 (LLM):")
display(per_dataset)
print("\nper-table mean F1 across runs:")
with pl.Config(tbl_rows=len(per_table)):
    display(per_table)